## Typicality rating experiment — LLaMA 3.1 8B Instruct

In [23]:
import os
# Must be set BEFORE importing transformers, or it will still probe TensorFlow
#os.environ["USE_TF"] = "0"
#os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import gc
import re
import pandas as pd
import torch
from transformers import pipeline

#MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/"  # change to "meta-llama/Meta-Llama-3-8B-Instruct" if you need 3.0

In [24]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

9355

In [25]:
df = pd.read_csv('combined_prototypes.csv')
df.head(5)

,category,concept_en,instance_English,instance_German,instance_Spanish,norm_rating_English,norm_rating_German,norm_rating_Spanish,n_languages,in_multiple_languages
0,animal,ANT,ant,Ameise,Hormiga,0.308662,0.630556,0.521127,3,True
1,animal,BEE,bee,Biene,Abeja,0.309422,0.711111,0.521127,3,True
2,animal,BUTTERFLY,butterfly,Schmetterling,Mariposa,0.298172,0.746667,0.507042,3,True
3,animal,CROW,crow,Krähe,Cuervo,0.499310,0.820556,0.394366,3,True
4,animal,DUCK,duck,Ente,Pato,0.772608,0.936667,0.830986,3,True


In [26]:
def typicality_prompt(obj, category):
    return f"""
You are participating in a psychology experiment.

Is "{obj}" a prototypical example of the category "{category}"?
Respond with ONLY one word: Yes or No.
"""

In [27]:
from transformers import pipeline

MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Device set to use cuda:0


In [28]:
import torch
import torch.nn.functional as F

model = pipe.model
tokenizer = pipe.tokenizer

# verify these are single tokens for your tokenizer (see check below)
yes_id = tokenizer.encode("Yes", add_special_tokens=False)[0]
no_id = tokenizer.encode("No", add_special_tokens=False)[0]

def get_typicality(obj, category):
    prompt = typicality_prompt(obj, category)
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=1,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
        )

    logits = out.scores[0][0]
    probs = F.softmax(logits, dim=-1)

    p_yes = probs[yes_id].item()
    p_no = probs[no_id].item()

    if (p_yes + p_no) < 1e-6:
        print(f"Warning: near-zero Yes/No mass for '{obj}' ({category}). Top tokens:")
        top = torch.topk(probs, 10)
        for val, idx in zip(top.values, top.indices):
            print(f"  {tokenizer.decode([idx])!r}: {val.item():.4f}")
        return None

    typicality_prob = p_yes / (p_yes + p_no)
    return typicality_prob

In [29]:
print(tokenizer.decode([yes_id]), tokenizer.decode([no_id]))

Yes No


In [ ]:
# Reset column in case a previous buggy run stored prompt text instead of scores
df["llama_typicality"] = pd.to_numeric(df["llama_typicality"], errors="coerce") if "llama_typicality" in df.columns else None

output_path = "combined_prototypes.csv"

for idx, row in df.iterrows():
    if pd.notna(df.at[idx, "llama_typicality"]):
        continue  # already done, skip (useful on resume)
    score = get_typicality(row["instance_English"], row["category"])
    df.at[idx, "llama_typicality"] = score
    df.to_csv(output_path, index=False)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end gene

In [ ]:
df.to_csv("llama_en_logprob.csv", index=False)
